# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through how to load, explore, and analyze the FAIR² Mass Spectrometry dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
This dataset is described via a [Croissant schema](https://mlcommons.org/croissant/) and made available at:
- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure the required library is installed
!pip install mlcroissant

## 1. Data Loading
Load the Croissant dataset and metadata using `mlcroissant`. The library provides convenient access to structured metadata and record sets.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access and print dataset-level metadata
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}\n")


## 2. Data Overview
Let us inspect available record sets and explore their fields. All entity references are by their `@id` fields.

**Note:** We will display all record sets (tables), the field `@id`s, their names, and the available columns.


In [ ]:
# List all record sets in the dataset, by @id
record_sets = dataset.metadata.recordSets
if not record_sets:
    print("No record sets found in this dataset metadata.")
else:
    print(f"Found {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        print(f"@id: {rs.id}")
        print(f"  Name: {rs.name}")
        print(f"  Description: {rs.description}")
        print("  Fields:")
        for fld in rs.fields:
            field_name = getattr(fld, 'name', '(none)')
            print(f"    - @id: {fld.id} | name: {field_name} | dataType: {getattr(fld, 'dataType', '(unknown)')}")
        print('')

### Preview records from a record set

Below we load and print several records from a record set for illustration. Please replace `<record_set_id>` with an actual `@id` from above (e.g. `'cr:RecordSet/0'`).


In [ ]:
# Choose a record set by @id (replace with actual @id printed above)
# Example: record_set_id = 'cr:RecordSet/0'
record_set_id = '<REPLACE_WITH_RECORD_SET_ID>'

try:
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        print(record)
        if i >= 2:  # print first 3 records
            break
except Exception as e:
    print(f"Error loading records: {e}\nMake sure to replace '<REPLACE_WITH_RECORD_SET_ID>' with a correct @id from the overview above.")

## 3. Data Extraction
Let's load all record sets into Pandas DataFrames, allowing for data exploration and analysis. Be sure to reference entities by their `@id`.


In [ ]:
# List all record set ids
record_sets_ids = [rs.id for rs in dataset.metadata.recordSets] if dataset.metadata.recordSets else []
dataframes = {}

for rs_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if len(records) > 0:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for {rs_id}: {dataframes[rs_id].shape[0]} records, {dataframes[rs_id].shape[1]} fields")
        else:
            print(f"No records found for record set {rs_id}.")
    except Exception as e:
        print(f"Failed to load record set {rs_id}: {e}")

# Choose a main record set for downstream EDA (edit as needed)
main_record_set_id = record_sets_ids[0] if record_sets_ids else None
if main_record_set_id:
    print(f"\nColumn list for {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Now, we perform exploratory data analysis and transformations using numeric fields. We'll filter, normalize, and group data. Please adjust `<numeric_field_id>` and `<group_field>` for your use-case by referencing the correct `@id` from the earlier overview.


In [ ]:
# These variable names should match the @id of the target field/column
numeric_field = '<REPLACE_WITH_NUMERIC_FIELD_ID>'  # e.g., 'coverage', 'peptide_counts', etc.
group_field = '<REPLACE_WITH_GROUP_FIELD_ID>'      # e.g., a sample type or condition field

df = dataframes.get(main_record_set_id)
if df is not None and numeric_field in df:
    # Filter for rows with values > threshold
    threshold = 10
    filtered_df = df[df[numeric_field].astype(float) > threshold]
    print(f"Filtered records where {numeric_field} > {threshold} (count: {filtered_df.shape[0]}):\n")
    print(filtered_df.head())

    # Normalize the numeric column
    filtered_df[numeric_field + '_normalized'] = (
        filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) / (
        filtered_df[numeric_field].astype(float).std() + 1e-9)
    print(f"\nNormalized {numeric_field} (mean=0, std=1):")
    print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

    # Group by another field, if present
    if group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped mean statistics by '{group_field}':")
        print(grouped_df.head())
    else:
        print(f"group_field '{group_field}' not found in columns.")
else:
    print(f"Column '{numeric_field}' not found in DataFrame or data unavailable. Be sure to set 'numeric_field' and 'group_field' by their @id as shown above.")

## 5. Visualization
Visualize distributions, correlations, or trends in the dataset. Replace variable names with actual `@id`s or field names as needed.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Histogram of a numeric field
if df is not None and numeric_field in df:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].astype(float), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

# Example: Boxplot grouped by a categorical field
if df is not None and numeric_field in df and group_field in df:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=df[group_field], y=df[numeric_field].astype(float))
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 6. Conclusion

In this notebook, we demonstrated how to load and explore a mass spectrometry dataset using `mlcroissant`. By referencing metadata and entities via their unique `@id`, we ensured reproducible, schema-driven data processing.

- We listed the dataset's record sets and fields via their `@id`.
- We loaded records into DataFrames and performed basic exploratory data analysis, such as filtering, normalization, and grouping.
- We visualized numeric distributions and relationships between data columns.

You can extend these analyses to other record sets, fields, or perform deeper domain-specific analyses as needed for your research workflows.